# 3. Enrich chunks and create Vector Search index

This notebook builds the **Gold** layer from `financial_db.filings.sec_rag_silver`. It adds FinBERT sentiment scores and BGE embeddings, writes the results to `financial_db.filings.sec_rag_gold`, then creates or synchronizes a Databricks Vector Search index for the app.

## Before running

- Run the ingestion and parsing notebooks first.
- Configure Databricks Connect and an identity with permission to read/write the catalog and create/manage Vector Search resources.
- Sentiment inference uses `ProsusAI/finbert`; embeddings use `BAAI/bge-small-en-v1.5`. The first run may download these models, and CPU execution may be slow.

## What the cells do

1. **Configure the Gold table.** Set the Silver input and Gold output names.

In [ ]:
import os
from databricks.connect import DatabricksSession
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pyspark.sql.functions import col
from sentence_transformers import SentenceTransformer
from databricks.ai_search.client import VectorSearchClient
from dotenv import load_dotenv

load_dotenv()

CATALOG = "financial_db"
SCHEMA = "filings"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.sec_rag_silver"
TABLE = f"{CATALOG}.{SCHEMA}.sec_rag_gold"

2. **Connect and prepare storage.** Start Databricks Connect and create the Gold table with sentiment and embedding columns if it does not exist.

In [ ]:
os.environ.pop("DATABRICKS_METADATA_SERVICE_URL", None)
os.environ["DATABRICKS_AUTH_TYPE"] = "pat"
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

spark = DatabricksSession.builder.getOrCreate()

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE} (
    chunk_id STRING,
    ticker STRING,
    cik STRING,
    date STRING,
    accession_number STRING,
    url STRING,
    chunk_index INT,
    chunk_text STRING,
    chunk_length INT,
    sentiment_label STRING,
    sentiment_pos FLOAT,
    sentiment_neg FLOAT,
    sentiment_neu FLOAT,
    embedding ARRAY<FLOAT>
)
USING DELTA TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
);
''')

3. **Enrich Silver chunks.** Run FinBERT to produce positive, negative, and neutral probabilities plus a label, generate normalized BGE embeddings, retain filing/chunk metadata, and append the enriched rows to Gold.

In [ ]:
def process_sentiment(text_series):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "ProsusAI/finbert"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    model.eval()
    
    labels = ["positive", "negative", "neutral"]
    results = []
    
    with torch.no_grad():
        texts = text_series.tolist()
        inputs = tokenizer(
            texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
        
        for prob in probs:
            pos, neg, neu = float(prob[0]), float(prob[1]), float(prob[2])
            top_label = labels[prob.argmax()]
            results.append({
                "label": top_label,
                "pos": pos,
                "neg": neg,
                "neu": neu
            })
            
    return results

def process_embeddings(text_series):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "BAAI/bge-small-en-v1.5"
    
    model = SentenceTransformer(model_name, device=device)
    texts = text_series.tolist()
    
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    
    return [emp.tolist() for emp in embeddings]

def get_gold():
    df_silver = spark.sql(f"""
        SELECT chunk_id, ticker, cik, date, accession_number, url, 
            chunk_index, chunk_text, chunk_length
        FROM {SILVER_TABLE}
        WHERE chunk_id NOT IN (SELECT DISTINCT chunk_id FROM {TABLE})
    """)

    if df_silver.count() > 0:
        df_pd = df_silver.toPandas()
        df_pd["sentiment"] = process_sentiment(df_pd["chunk_text"])
        df_pd["embedding"] = process_embeddings(df_pd["chunk_text"])
        df = spark.createDataFrame(df_pd)
        df = df.select(
                col("chunk_id"),
                col("ticker"),
                col("cik"),
                col("date"),
                col("accession_number"),
                col("url"),
                col("chunk_index"),
                col("chunk_text"),
                col("chunk_length"),
                col("sentiment.label").alias("sentiment_label"),
                col("sentiment.pos").alias("sentiment_pos"),
                col("sentiment.neg").alias("sentiment_neg"),
                col("sentiment.neu").alias("sentiment_neu"),
                col("embedding")
            )
        df.write.format("delta").mode("append").saveAsTable(TABLE)

get_gold()

4. **Create or synchronize Vector Search.** Ensure the configured endpoint exists, then synchronize the existing index or create a triggered Delta Sync index using `chunk_id` as its key and the stored 384-dimensional embedding vector.

In [ ]:
vsc = VectorSearchClient(disable_notice=True)

endpoint_name = "financial_rag_endpoint"

existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]

if endpoint_name not in existing_endpoints:
    vsc.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

index_name = f"{CATALOG}.{SCHEMA}.sec_gold_vector_index"

try:
    index = vsc.get_index(endpoint_name=endpoint_name, index_name=index_name)
    index.sync()
except Exception:
    index = vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=TABLE,
        index_name=index_name,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_dimension=384,
        embedding_vector_column="embedding"
    )

5. **Inspect results.** Preview the highest positive-sentiment rows, their vector dimensions, and a short text excerpt.

In [ ]:
spark.sql(f"""
    SELECT ticker, date, chunk_index, sentiment_label, 
           round(sentiment_pos, 2) as pos, round(sentiment_neg, 2) as neg,
           size(embedding) as vector_dim, substring(chunk_text, 1, 80) as text_snippet
    FROM {TABLE}
    ORDER BY sentiment_pos DESC
    LIMIT 10
""").toPandas()